# TB Portals — Published Shift-Robust Regression Baselines

Implements two **published** shift-robust regression methods on locked TorchXRayVision DenseNet121 features and produces Timika predictions on our LOCO test set:

1. **GroupDRO** (Sagawa et al., *ICLR 2020*) — worst-group-loss minimisation, groups = source countries.
2. **Importance-Weighted Regression** (Shimodaira 2000; Sugiyama et al. 2007) — re-weight training loss by estimated $P_{\text{tgt}}(x) / P_{\text{src}}(x)$ via Gaussian-KDE on the locked feature space.

Both methods train on the source train pool, are evaluated on each held-out country, 5 seeds. Per-image preds are saved for paired-bootstrap comparison against our pipeline.

Attach `tb-portals-cxr-pngs`. Internet **ON**. GPU T4. Runtime ≈ 3 hr.

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH = 'cleaned-repo'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torchxrayvision', 'transformers', 'scikit-learn'], check=False)
print('deps installed')

In [ ]:
import os, pandas as pd, numpy as np, torch
from pathlib import Path
WORK = '/kaggle/working'
REPO_DIR = '/kaggle/working/dl-project-codebase'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'
PAPER_MANIFEST = f'{WORK}/tbportals_manifest_paper.csv'
FEATURES_TXV  = f'{WORK}/features_txv_cls.npz'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')

from build_paper_manifest import subsample
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
paper_df = subsample(raw, seed=42)
paper_df['image_id'] = paper_df['image_path'].apply(lambda p: Path(str(p)).stem)
paper_df['timika'] = paper_df['alp_0_100'] + 40 * paper_df['cavity']
paper_df.to_csv(PAPER_MANIFEST, index=False)

from cache_features import main as cache_main, load_features
if not os.path.isfile(FEATURES_TXV):
    cache_main(['--manifest', PAPER_MANIFEST, '--out', FEATURES_TXV,
                '--backbone', 'txrv', '--batch-size', '32'])
feats, dim = load_features(FEATURES_TXV)
print('TXV features:', dim, '| coverage:', sum(1 for i in paper_df['image_id'] if i in feats), '/', len(paper_df))

## 1 — Shared training utilities (R1-style 2-layer MLP head)

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from src.data.tbportals import make_country_split

class Head(nn.Module):
    def __init__(self, in_dim, hidden=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden, 1))
    def forward(self, x):
        return torch.sigmoid(self.net(x)).squeeze(1)  # [B] in [0,1]

def get_xy(df, feats, dim):
    df = df[df['image_id'].isin(feats)].copy()
    X = np.stack([feats[i] for i in df['image_id']]).astype(np.float32)
    y = (df['alp_0_100'].values + 40 * df['cavity'].values).astype(np.float32) / 140.0
    return X, y, df

def make_groups(df, train_countries):
    """Group label = country index (0/1) in train_countries."""
    m = {c: i for i, c in enumerate(train_countries)}
    return df['country'].map(m).values.astype(np.int64)

def predict(model, X):
    model.eval()
    with torch.no_grad():
        x = torch.from_numpy(X).to(device)
        return (model(x).cpu().numpy() * 140.0)

EPOCHS = 50
LR = 1e-3
BATCH = 256

## 2 — GroupDRO baseline

In [ ]:
def train_groupdro(X_tr, y_tr, g_tr, in_dim, n_groups, *, eta=0.01, seed=0):
    """Sagawa et al. 2020 -- min over theta of max over groups of E[loss | group].
    Adversary maintains a simplex over group weights; update via multiplicative weights."""
    torch.manual_seed(seed)
    model = Head(in_dim).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    q = torch.ones(n_groups, device=device) / n_groups
    N = X_tr.shape[0]
    for epoch in range(EPOCHS):
        idx = np.random.RandomState(seed * 10000 + epoch).permutation(N)
        for s in range(0, N, BATCH):
            sl = idx[s:s + BATCH]
            xb = torch.from_numpy(X_tr[sl]).to(device)
            yb = torch.from_numpy(y_tr[sl]).to(device)
            gb = torch.from_numpy(g_tr[sl]).to(device)
            pred = model(xb)
            per_sample = (pred - yb).pow(2)
            group_losses = torch.stack([per_sample[gb == k].mean() if (gb == k).any() else torch.tensor(0.0, device=device)
                                        for k in range(n_groups)])
            # Update adversary q (exponentiated gradient)
            with torch.no_grad():
                q = q * torch.exp(eta * group_losses.detach())
                q = q / q.sum()
            loss = (q * group_losses).sum()
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def train_iw_regression(X_tr, y_tr, X_te, in_dim, *, seed=0):
    """Importance-weighted regression (Shimodaira 2000).
    Density-ratio P_tgt(x) / P_src(x) estimated by RBF-kernel ratio on a random sub-sample.
    """
    torch.manual_seed(seed)
    rng = np.random.RandomState(seed)
    # subsample for tractable kernel comp
    K = 256
    idx_s = rng.choice(X_tr.shape[0], min(K, X_tr.shape[0]), replace=False)
    idx_t = rng.choice(X_te.shape[0], min(K, X_te.shape[0]), replace=False)
    # robust median bandwidth
    diffs = X_tr[idx_s][:, None, :] - X_tr[idx_s][None, :, :]
    sigma = max(float(np.sqrt(np.median((diffs ** 2).sum(axis=-1)))), 1.0)
    def kgauss(A, B):
        D = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1)
        return np.exp(-D / (2 * sigma * sigma))
    # density ratio: p_tgt(x) ~ avg kernel to target sample; p_src(x) ~ avg kernel to source sample
    Ks_to_t = kgauss(X_tr, X_te[idx_t]).mean(axis=1)  # p_tgt at each train point
    Ks_to_s = kgauss(X_tr, X_tr[idx_s]).mean(axis=1)  # p_src at each train point
    w = (Ks_to_t / (Ks_to_s + 1e-8)).astype(np.float32)
    w = np.clip(w, 0.1, 10.0)
    w = w / w.mean()  # normalise to mean 1
    print(f'IW weights: min {w.min():.2f}, max {w.max():.2f}, mean {w.mean():.2f}')

    model = Head(in_dim).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    N = X_tr.shape[0]
    for epoch in range(EPOCHS):
        idx = np.random.RandomState(seed * 10000 + epoch).permutation(N)
        for s in range(0, N, BATCH):
            sl = idx[s:s + BATCH]
            xb = torch.from_numpy(X_tr[sl]).to(device)
            yb = torch.from_numpy(y_tr[sl]).to(device)
            wb = torch.from_numpy(w[sl]).to(device)
            pred = model(xb)
            per_sample = (pred - yb).pow(2)
            loss = (wb * per_sample).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return model

## 3 — Run both methods × 3 countries × 5 seeds

In [ ]:
all_rows = []
for country in ['Romania', 'Moldova', 'Kazakhstan']:
    for seed in range(5):
        tr_df, _, te_df = make_country_split(paper_df, held_out_country=country, val_fraction=0.2, seed=seed)
        X_tr, y_tr, tr_used = get_xy(tr_df, feats, dim)
        X_te, y_te, te_used = get_xy(te_df, feats, dim)
        train_countries = sorted(tr_used['country'].unique().tolist())
        g_tr = make_groups(tr_used, train_countries)

        # GroupDRO
        model_g = train_groupdro(X_tr, y_tr, g_tr, in_dim=dim,
                                  n_groups=len(train_countries), seed=seed)
        yhat_g = predict(model_g, X_te)
        mae_g = float(np.mean(np.abs(yhat_g - y_te * 140.0)))
        # IW
        model_iw = train_iw_regression(X_tr, y_tr, X_te, in_dim=dim, seed=seed)
        yhat_iw = predict(model_iw, X_te)
        mae_iw = float(np.mean(np.abs(yhat_iw - y_te * 140.0)))
        print(f'{country} s{seed}: GroupDRO MAE = {mae_g:.2f} | IW MAE = {mae_iw:.2f}')

        for img_id, gt, ph_g, ph_iw in zip(te_used['image_id'].values, y_te * 140.0, yhat_g, yhat_iw):
            all_rows.append({'image_id': img_id, 'held_out': country, 'seed': seed,
                             'method': 'GroupDRO (TXV)', 'timika_true': float(gt),
                             'timika_pred': float(ph_g)})
            all_rows.append({'image_id': img_id, 'held_out': country, 'seed': seed,
                             'method': 'Importance-Weighted (TXV)', 'timika_true': float(gt),
                             'timika_pred': float(ph_iw)})

out = pd.DataFrame(all_rows)
out.to_csv(f'{WORK}/preds_shift_robust_baselines.csv', index=False)
summary = (out.groupby(['method', 'held_out', 'seed']).apply(
             lambda d: float(np.mean(np.abs(d['timika_pred'] - d['timika_true'])))
           ).reset_index(name='timika_mae'))
agg = summary.groupby(['method', 'held_out'])['timika_mae'].agg(['mean','std']).round(2)
print(agg)
summary.to_csv(f'{WORK}/summary_shift_robust.csv', index=False)

In [ ]:
import shutil
OUT = f'{WORK}/shift_robust_baselines'
os.makedirs(OUT, exist_ok=True)
for f in [f'{WORK}/preds_shift_robust_baselines.csv', f'{WORK}/summary_shift_robust.csv']:
    shutil.copy(f, OUT)
zip_path = shutil.make_archive(f'{WORK}/shift_robust_baselines', 'zip', OUT)
print('zip ->', zip_path)